# IMC Prosperity 4 – Market Data Analysis

**Purpose:** Extract actionable trading insights from Prosperity price and trade CSVs.

**Usage:**
- Drop any new `prices_round_*_day_*.csv` / `trades_round_*_day_*.csv` files into this folder and re-run all cells.
- Sections build on each other — run top to bottom on first pass.

---
| Section | Topic |
|---------|-------|
| 1 | Data Loading |
| 2 | Price & Fair Value Analysis |
| 3 | Spread Analysis |
| 4 | Mean Reversion Test |
| 5 | Signal Analysis |
| 6 | Order Book Depth |
| 7 | Market Trades |
| 8 | Position Limit Stress Test |
| 9 | Product Correlation |
| 10 | Strategy Recommendation Summary |

In [ ]:
# ── shared imports & helpers ─────────────────────────────────────────────
import os, glob, warnings, statistics
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.dpi'] = 110

DAY_COLOURS = ['steelblue', 'darkorange', 'green', 'red', 'purple', 'brown']

def compute_microprice(df):
    """Microprice = vol-weighted mid (tilts toward the thinner side of the book)."""
    num = df['bid_price_1'] * df['ask_volume_1'] + df['ask_price_1'] * df['bid_volume_1']
    den = df['bid_volume_1'] + df['ask_volume_1']
    return num / den

def autocorr_at_lag(series, lag):
    s = series.dropna()
    if len(s) <= lag: return float('nan')
    return float(np.corrcoef(s[lag:], s[:-lag])[0, 1])

def signal_stats(signal, next_ret, name):
    mask = signal.notna() & next_ret.notna()
    s, r = signal[mask], next_ret[mask]
    if len(s) < 10: return {'name': name, 'corr': float('nan'), 'hit_rate': float('nan')}
    corr     = float(np.corrcoef(s, r)[0, 1])
    hits     = ((np.sign(s) == np.sign(r)) & (r != 0)).sum()
    total    = (r != 0).sum()
    hit_rate = hits / total if total > 0 else float('nan')
    return {'name': name, 'corr': corr, 'hit_rate': hit_rate}

print('Imports OK')

: 

---
## Section 1 — Data Loading

Scans the current directory for all matching CSVs, adds `round`/`day` tags, and merges into unified dataframes.
The `products` dict maps each product name to its price dataframe — all later sections use this.

In [ ]:
# ── load prices ──────────────────────────────────────────────────────────
def load_prices():
    files = sorted(glob.glob('prices_round_*_day_*.csv'))
    if not files:
        raise FileNotFoundError('No prices_round_*_day_*.csv files found.')
    frames = []
    for f in files:
        p = f.replace('.csv','').split('_')
        rnd, day = int(p[2]), int(p[4])
        df = pd.read_csv(f, sep=';')
        df['round'], df['day'], df['sort_key'] = rnd, day, rnd*1000+day
        frames.append(df)
        print(f'  {f}  →  {len(df):,} rows  (round={rnd}, day={day})')
    out = pd.concat(frames, ignore_index=True)
    out.sort_values(['sort_key','timestamp','product'], inplace=True)
    out.reset_index(drop=True, inplace=True)
    return out

def load_trades():
    files = sorted(glob.glob('trades_round_*_day_*.csv'))
    if not files:
        print('WARNING: no trades CSVs found.')
        return pd.DataFrame()
    frames = []
    for f in files:
        p = f.replace('.csv','').split('_')
        rnd, day = int(p[2]), int(p[4])
        df = pd.read_csv(f, sep=';')
        df['round'], df['day'], df['sort_key'] = rnd, day, rnd*1000+day
        frames.append(df)
        print(f'  {f}  →  {len(df):,} rows  (round={rnd}, day={day})')
    out = pd.concat(frames, ignore_index=True)
    out.sort_values(['sort_key','timestamp'], inplace=True)
    out.reset_index(drop=True, inplace=True)
    return out

prices = load_prices()
trades = load_trades()
products = {p: prices[prices['product']==p].copy() for p in sorted(prices['product'].unique())}
print(f'\nProducts loaded: {list(products.keys())}')

In [ ]:
# ── summary table ────────────────────────────────────────────────────────
rows = []
for prod, df in products.items():
    spread = (df['ask_price_1'] - df['bid_price_1'])
    rows.append({
        'Product'    : prod,
        'Days'       : str(sorted(df['day'].unique())),
        'Ticks'      : len(df),
        'Price Min'  : df['mid_price'].min(),
        'Price Max'  : df['mid_price'].max(),
        'Mean Spread': round(spread.mean(), 4),
    })
summary = pd.DataFrame(rows).set_index('Product')
print('=== DATA SUMMARY ===')
print(summary.to_string())
if not trades.empty:
    print(f'\nTrade rows: {len(trades):,}  |  Symbols: {sorted(trades["symbol"].unique())}')

---
## Section 2 — Price & Fair Value Analysis

**Simple mid** = `(bid_price_1 + ask_price_1) / 2` — equal-weight average of best quotes.  
**Microprice** tilts the fair value toward whichever side has *less* size:
$$\text{microprice} = \frac{\text{bid}_1 \times \text{ask\_vol}_1 + \text{ask}_1 \times \text{bid\_vol}_1}{\text{bid\_vol}_1 + \text{ask\_vol}_1}$$
When ask is thin, microprice > mid → price likely to move up soon.  

**Trading implication:** Use microprice as a short-term fair value signal; if microprice > mid, prefer to buy rather than sell.

In [ ]:
for prod, df in products.items():
    days = sorted(df['day'].unique())
    fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=False)
    fig.suptitle(f'{prod} – Price & Fair Value Analysis', fontsize=14, fontweight='bold')

    # panel 1: mid_price per day
    ax = axes[0]
    for i, day in enumerate(days):
        d = df[df['day']==day]
        ax.plot(d['timestamp'], d['mid_price'], color=DAY_COLOURS[i%len(DAY_COLOURS)],
                lw=0.8, label=f'Day {day}', alpha=0.9)
    ax.set_title('Mid Price (all days)')
    ax.set_ylabel('Price'); ax.legend(); ax.grid(True, alpha=0.3)

    # panel 2: simple mid vs microprice (first day)
    ax = axes[1]
    d2  = df[df['day']==days[0]].copy()
    ax.plot(d2['timestamp'], (d2['bid_price_1']+d2['ask_price_1'])/2,
            color='steelblue', lw=0.8, label='Simple Mid', alpha=0.8)
    ax.plot(d2['timestamp'], compute_microprice(d2),
            color='crimson',   lw=0.8, label='Microprice',  alpha=0.8)
    ax.set_title(f'Simple Mid vs Microprice – Day {days[0]}')
    ax.set_ylabel('Price'); ax.legend(); ax.grid(True, alpha=0.3)

    # panel 3: rolling means
    ax = axes[2]
    for i, day in enumerate(days):
        d = df[df['day']==day].copy()
        col = DAY_COLOURS[i%len(DAY_COLOURS)]
        ax.plot(d['timestamp'], d['mid_price'], color=col, lw=0.5, alpha=0.35)
        for w, ls in [(10,'-'),(20,'--'),(50,':')]:
            ax.plot(d['timestamp'], d['mid_price'].rolling(w).mean(),
                    color=col, lw=1.2, ls=ls,
                    label=f'Day {day} RM{w}' if i==0 else None)
    ax.set_title('Rolling Means (10 / 20 / 50 ticks)')
    ax.set_xlabel('Timestamp'); ax.set_ylabel('Price')
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

    plt.tight_layout(); plt.show()

    mp  = df['mid_price']
    std = mp.std()
    print(f'{prod}: mean={mp.mean():.4f}  std={std:.4f}  '
          f'min={mp.min():.2f}  max={mp.max():.2f}  '
          f'→ {"STATIONARY" if std < 5 else "DRIFTING"}\n')

---
## Section 3 — Spread Analysis

The bid-ask spread is the *baseline cost* of crossing the market and the *baseline revenue* of providing liquidity.  
- **Market maker profit** ≈ ½ × spread per side (assuming zero adverse selection).  
- **Wide spread** → bigger edge but fewer fill opportunities.  
- **Narrow spread** → tighter edge; need higher volume to break even on inventory risk.

The histogram shows the spread distribution — if it's bimodal you may be seeing regime changes (e.g. open vs. normal session).

In [ ]:
for prod, df in products.items():
    days = sorted(df['day'].unique())
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'{prod} – Spread Analysis', fontsize=14, fontweight='bold')

    ax = axes[0]
    for i, day in enumerate(days):
        d  = df[df['day']==day]
        sp = d['ask_price_1'] - d['bid_price_1']
        ax.plot(d['timestamp'], sp, color=DAY_COLOURS[i%len(DAY_COLOURS)],
                lw=0.7, label=f'Day {day}', alpha=0.85)
    ax.set_title('Bid-Ask Spread Over Time'); ax.set_xlabel('Timestamp')
    ax.set_ylabel('Spread (ticks)'); ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[1]
    sp_all = df['ask_price_1'] - df['bid_price_1']
    ax.hist(sp_all, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(sp_all.mean(), color='red', lw=1.5, ls='--',
               label=f'Mean {sp_all.mean():.2f}')
    ax.set_title('Spread Distribution'); ax.set_xlabel('Spread (ticks)')
    ax.set_ylabel('Frequency'); ax.legend(); ax.grid(True, alpha=0.3)

    plt.tight_layout(); plt.show()

    w_thr = sp_all.mean() + 2*sp_all.std()
    n_thr = sp_all.mean() -   sp_all.std()
    print(f'{prod}:')
    print(f'  Mean={sp_all.mean():.4f}  Median={float(np.median(sp_all)):.4f}  '
          f'Std={sp_all.std():.4f}  Min={sp_all.min():.1f}  Max={sp_all.max():.1f}')
    print(f'  Wide (>{w_thr:.1f}): {(sp_all>w_thr).mean()*100:.2f}%  '
          f'Narrow (<{n_thr:.1f}): {(sp_all<n_thr).mean()*100:.2f}%')
    print(f'  Theoretical round-trip profit (½ spread): {sp_all.mean()/2:.4f}\n')

---
## Section 4 — Mean Reversion Test

**Return autocorrelation at lag k** measures whether the price move at tick *t* predicts the move at tick *t+k*:
- **Negative lag-1 AC** → mean reverting: a up-tick is likely followed by a down-tick. Strategy: fade moves.
- **Positive lag-1 AC** → trending: momentum persists. Strategy: follow moves.
- Near zero → random walk.

**Z-score plot:** when |z| > 2 the price is an extreme deviation from its local mean — a potential entry for a mean-reversion trade.  
> Threshold |z| > 1.5–2 is a common entry trigger; exit when |z| < 0.5.

In [ ]:
LAGS = [1, 2, 3, 5, 10]

for prod, df in products.items():
    days = sorted(df['day'].unique())
    d0 = df[df['day']==days[0]].copy()
    d0['ret']  = d0['mid_price'].diff()
    d0['rm20'] = d0['mid_price'].rolling(20).mean()
    d0['std20']= d0['mid_price'].rolling(20).std()
    d0['zscore']= (d0['mid_price']-d0['rm20']) / d0['std20']

    rets_all = df.groupby('day')['mid_price'].diff().dropna()
    ac = [autocorr_at_lag(rets_all, lag) for lag in LAGS]

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(f'{prod} – Mean Reversion Analysis', fontsize=14, fontweight='bold')

    ax = axes[0]
    ax.plot(d0['timestamp'], d0['zscore'], color='steelblue', lw=0.7)
    for h, col in [(0,'black'),(2,'red'),(-2,'red')]:
        ax.axhline(h, color=col, lw=1, ls='--' if h!=0 else '-')
    ax.set_title('Z-Score (RM20)'); ax.set_xlabel('Timestamp'); ax.set_ylabel('Z')
    ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(d0['timestamp'], d0['mid_price'], color='steelblue', lw=0.7,
            label='Mid', alpha=0.7)
    ax.plot(d0['timestamp'], d0['rm20'], color='red', lw=1.2, label='RM20')
    ax.set_title(f'Mid vs RM20 – Day {days[0]}')
    ax.set_xlabel('Timestamp'); ax.set_ylabel('Price')
    ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[2]
    colours = ['red' if v<0 else 'steelblue' for v in ac]
    ax.bar([str(l) for l in LAGS], ac, color=colours, edgecolor='white', alpha=0.8)
    ax.axhline(0, color='black', lw=1)
    ax.set_title('Return Autocorrelation by Lag')
    ax.set_xlabel('Lag (ticks)'); ax.set_ylabel('Correlation')
    ax.grid(True, alpha=0.3)

    plt.tight_layout(); plt.show()

    lag1 = ac[0]
    verdict = 'MEAN REVERTING' if lag1 < -0.05 else ('TRENDING' if lag1 > 0.05 else 'MIXED')
    print(f'{prod}: lag-1 AC={lag1:.4f}  →  *** {verdict} ***')
    print(f'  All lags: {dict(zip(LAGS, [f"{v:.4f}" for v in ac]))}\n')

---
## Section 5 — Signal Analysis

Tests three candidate signals for predicting the **next-tick price direction**:

| Signal | Formula | Hypothesis |
|--------|---------|------------|
| `ret1` | `mid[t] - mid[t-1]` | Negative corr → mean revert; positive → momentum |
| `z20`  | `mid[t] - RM20[t]`  | Negative corr → mean revert to 20-tick window |
| `microprice_delta` | `microprice[t] - mid[t]` | Positive corr → order flow imbalance leads price |

**Hit rate > 50%** means the signal directionally predicts more often than random.  
**Correlation** tells you the *magnitude* of predictive power — use as a signal weight.

In [ ]:
for prod, df in products.items():
    d = df.copy()
    d['ret']         = d['mid_price'].diff()
    d['rm20']        = d['mid_price'].rolling(20).mean()
    d['z20']         = d['mid_price'] - d['rm20']
    d['micro']       = compute_microprice(d)
    d['micro_delta'] = d['micro'] - d['mid_price']
    d['next_ret']    = d['ret'].shift(-1)

    sig1 = signal_stats(d['ret'],         d['next_ret'], 'ret1 (momentum/reversal)')
    sig2 = signal_stats(d['z20'],         d['next_ret'], 'z20  (mean reversion)')
    sig3 = signal_stats(d['micro_delta'], d['next_ret'], 'microprice delta')

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(f'{prod} – Signal vs Next-Tick Return', fontsize=14, fontweight='bold')

    for ax, (sig_s, meta) in zip(axes, [
        (d['ret'],         sig1),
        (d['z20'],         sig2),
        (d['micro_delta'], sig3),
    ]):
        mask = sig_s.notna() & d['next_ret'].notna()
        xs, ys = sig_s[mask].values, d['next_ret'][mask].values
        ax.scatter(xs, ys, s=2, alpha=0.3, color='steelblue')
        if len(xs) > 2:
            m, b = np.polyfit(xs, ys, 1)
            xr = np.linspace(xs.min(), xs.max(), 100)
            ax.plot(xr, m*xr+b, color='red', lw=1.5)
        ax.set_title(f"{meta['name']}\ncorr={meta['corr']:.4f}  hit%={meta['hit_rate']*100:.1f}%")
        ax.set_xlabel('Signal'); ax.set_ylabel('Next tick return')
        ax.grid(True, alpha=0.3)

    plt.tight_layout(); plt.show()

    print(f'{prod}:')
    for s in [sig1, sig2, sig3]:
        c = s['corr']
        w = ('CONTRARIAN (mean-reversion)' if c < -0.01 else
             'MOMENTUM' if c > 0.01 else 'NO EDGE')
        print(f"  {s['name']:30s}  corr={c:+.4f}  hit%={s['hit_rate']*100:.1f}%  → {w}")
    print()

---
## Section 6 — Order Book Depth Analysis

**Book imbalance** = `(bid_vol - ask_vol) / (bid_vol + ask_vol)`  
- +1 → all volume on bid side → buyers are aggressive → price likely to go UP
- -1 → all volume on ask side → sellers are aggressive → price likely to go DOWN

**Level analysis:** seeing most volume at L1 means the order book is shallow — large orders move price quickly. Seeing deep L2/L3 means the book absorbs more before price impacts.

> **Trading use:** If imbalance strongly predicts direction, it can be a low-latency execution trigger — tilt quotes or hit passively before the move.

In [ ]:
for prod, df in products.items():
    days = sorted(df['day'].unique())
    d0 = df[df['day']==days[0]].copy()

    for vc in ['bid_volume_1','bid_volume_2','bid_volume_3',
               'ask_volume_1','ask_volume_2','ask_volume_3']:
        d0[vc] = d0[vc].fillna(0)

    d0['total_bid'] = d0[['bid_volume_1','bid_volume_2','bid_volume_3']].sum(axis=1)
    d0['total_ask'] = d0[['ask_volume_1','ask_volume_2','ask_volume_3']].sum(axis=1)
    d0['imbalance'] = (d0['total_bid']-d0['total_ask']) / (d0['total_bid']+d0['total_ask'])
    d0['next_ret']  = d0['mid_price'].diff().shift(-1)

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle(f'{prod} – Order Book Depth (Day {days[0]})',
                 fontsize=14, fontweight='bold')

    axes[0,0].plot(d0['timestamp'], d0['total_bid'], color='green', lw=0.8, label='Bid')
    axes[0,0].plot(d0['timestamp'], d0['total_ask'], color='red',   lw=0.8, label='Ask')
    axes[0,0].set_title('Total Bid vs Ask Volume')
    axes[0,0].set_xlabel('Timestamp'); axes[0,0].set_ylabel('Volume')
    axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

    axes[0,1].plot(d0['timestamp'], d0['imbalance'], color='steelblue', lw=0.6)
    axes[0,1].axhline(0, color='black', lw=1)
    axes[0,1].set_title('Book Imbalance  (+1=all-bid, -1=all-ask)')
    axes[0,1].set_xlabel('Timestamp'); axes[0,1].set_ylabel('Imbalance')
    axes[0,1].grid(True, alpha=0.3)

    for lv, col in [(1,'steelblue'),(2,'darkorange'),(3,'green')]:
        c = f'bid_volume_{lv}'
        if c in d0.columns:
            axes[1,0].fill_between(d0['timestamp'], d0[c], alpha=0.5,
                                   label=f'L{lv}', color=col)
    axes[1,0].set_title('Bid Volume by Level')
    axes[1,0].set_xlabel('Timestamp'); axes[1,0].set_ylabel('Volume')
    axes[1,0].legend(); axes[1,0].grid(True, alpha=0.3)

    for lv, col in [(1,'steelblue'),(2,'darkorange'),(3,'green')]:
        c = f'ask_volume_{lv}'
        if c in d0.columns:
            axes[1,1].fill_between(d0['timestamp'], d0[c], alpha=0.5,
                                   label=f'L{lv}', color=col)
    axes[1,1].set_title('Ask Volume by Level')
    axes[1,1].set_xlabel('Timestamp'); axes[1,1].set_ylabel('Volume')
    axes[1,1].legend(); axes[1,1].grid(True, alpha=0.3)

    plt.tight_layout(); plt.show()

    res = signal_stats(d0['imbalance'], d0['next_ret'], 'imbalance')
    print(f'{prod}:  imbalance→next_ret  corr={res["corr"]:.4f}  '
          f'hit%={res["hit_rate"]*100:.2f}%\n')

---
## Section 7 — Market Trades Analysis

Market trades reveal *who is taking liquidity*:
- **Buyer-initiated** (trade price ≥ mid): aggressive buyer lifted the ask → bullish signal
- **Seller-initiated** (trade price < mid): aggressive seller hit the bid → bearish signal

**Late-day pattern:** In Prosperity the timestamp runs to 999,900. If >80% of trades occur after ts=140,000 it means the late-hold strategy is relevant — the bot should manage flat (or close positions) early and let inventory accumulate into the session end.

> Watch for **trade clusters**: 3+ trades in a short window often precede a directional move as one large participant works through their order.

In [ ]:
if trades.empty:
    print('No trade data available – skipping section 7.')
else:
    sym_col = 'symbol' if 'symbol' in trades.columns else 'product'

    for prod, price_df in products.items():
        t = trades[trades[sym_col]==prod].copy()
        if t.empty:
            print(f'{prod}: no trades found.'); continue

        days = sorted(price_df['day'].unique())
        mid_lookup = price_df[['day','timestamp','mid_price',
                                'bid_price_1','ask_price_1']].copy()

        for day in days:
            td = t[t['day']==day].copy()
            md = mid_lookup[mid_lookup['day']==day].copy()
            if len(td) > 0 and len(md) > 0: break
        else:
            print(f'{prod}: no matching day.'); continue

        td = td.sort_values('timestamp')
        md = md.sort_values('timestamp')
        td = pd.merge_asof(td, md[['timestamp','mid_price','bid_price_1','ask_price_1']],
                           on='timestamp', direction='backward')
        td['is_buy'] = td['price'] >= td['mid_price']

        bins = range(int(td['timestamp'].min()), int(td['timestamp'].max())+1001, 1000)
        td['ts_bin'] = pd.cut(td['timestamp'], bins=list(bins), labels=False)
        freq = td.groupby('ts_bin').size()

        fig, axes = plt.subplots(2, 2, figsize=(16, 10))
        fig.suptitle(f'{prod} – Market Trades (Day {day})',
                     fontsize=14, fontweight='bold')

        axes[0,0].plot(md['timestamp'], md['mid_price'], color='steelblue',
                       lw=0.8, label='Mid', alpha=0.7)
        buy_t  = td[td['is_buy']]
        sell_t = td[~td['is_buy']]
        axes[0,0].scatter(buy_t['timestamp'],  buy_t['price'],  s=12, color='green',
                          zorder=3, label='Buy')
        axes[0,0].scatter(sell_t['timestamp'], sell_t['price'], s=12, color='red',
                          zorder=3, label='Sell')
        axes[0,0].set_title('Trade Prices vs Mid')
        axes[0,0].set_xlabel('Timestamp'); axes[0,0].set_ylabel('Price')
        axes[0,0].legend(fontsize=8); axes[0,0].grid(True, alpha=0.3)

        axes[0,1].bar(freq.index, freq.values, color='steelblue', alpha=0.8)
        axes[0,1].set_title('Trade Frequency (per 1000-ts bin)')
        axes[0,1].set_xlabel('Bin'); axes[0,1].set_ylabel('# Trades')
        axes[0,1].grid(True, alpha=0.3)

        if 'quantity' in td.columns:
            td['quantity'] = pd.to_numeric(td['quantity'], errors='coerce')
            axes[1,0].hist(td['quantity'].dropna(), bins=20,
                           color='darkorange', edgecolor='white', alpha=0.8)
        axes[1,0].set_title('Trade Size Distribution')
        axes[1,0].set_xlabel('Quantity'); axes[1,0].set_ylabel('Frequency')
        axes[1,0].grid(True, alpha=0.3)

        early = td[td['timestamp']<=140_000]
        late  = td[td['timestamp'] >140_000]
        axes[1,1].bar(['Early (≤140k)','Late (>140k)'], [len(early),len(late)],
                      color=['steelblue','darkorange'], alpha=0.8)
        axes[1,1].set_title('Early vs Late Session Trades')
        axes[1,1].set_ylabel('# Trades'); axes[1,1].grid(True, alpha=0.3)

        plt.tight_layout(); plt.show()

        ratio = len(late)/max(len(early),1)
        note  = (f'Late session MORE active ({ratio:.1f}×) → late-hold strategy relevant'
                 if ratio > 1.2 else
                 f'Late session LESS active ({ratio:.1f}×) → early action dominates'
                 if ratio < 0.8 else 'Activity roughly even')
        print(f'{prod} (day {day}): {len(td)} trades  '
              f'buy={len(buy_t)/len(td)*100:.1f}%  sell={len(sell_t)/len(td)*100:.1f}%')
        print(f'  Early={len(early)}  Late={len(late)}  → {note}\n')

---
## Section 8 — Position Limit Stress Test

Simulates a simple market-making strategy against the historical data:
- **BUY** when `ask_price_1 ≤ fair_value − take_edge`
- **SELL** when `bid_price_1 ≥ fair_value + take_edge`
- Position capped at ±80 (Prosperity Round 1 limit)

**What to look for:**
- If position keeps hitting the limit → `take_edge` is too tight, reduce aggression or add soft-cap inventory penalty.
- If position never approaches the limit → `take_edge` may be too wide; you're leaving edge on the table.
- Long average unwind time → you may need to actively unwind at a loss; factor this into P&L estimates.

This directly informs the `TOMATO_WARNING` and `TOMATO_SOFT_CAP` tuning parameters.

In [ ]:
def simulate_mm(price_df, fair_value_func, take_edge=2.0, pos_limit=80):
    positions, pos = [], 0
    for _, row in price_df.iterrows():
        fv   = fair_value_func(row)
        bid1 = row['bid_price_1']; ask1 = row['ask_price_1']
        bv1  = row.get('bid_volume_1',0) or 0
        av1  = row.get('ask_volume_1',0) or 0
        if ask1 <= fv - take_edge and pos < pos_limit:
            pos += min(int(av1), pos_limit - pos)
        elif bid1 >= fv + take_edge and pos > -pos_limit:
            pos -= min(int(bv1), pos_limit + pos)
        positions.append(pos)
    return pd.Series(positions, index=price_df.index)

POS_LIMIT = 80
fv_fns = {
    'EMERALDS': lambda row: 10_000.0,
    'TOMATOES': lambda row: row['mid_price'],
}

for prod, df in products.items():
    days  = sorted(df['day'].unique())
    fv_fn = fv_fns.get(prod, lambda row: row['mid_price'])

    fig, axes = plt.subplots(len(days), 1, figsize=(14, 5*len(days)), squeeze=False)
    fig.suptitle(f'{prod} – Position Simulation (limit={POS_LIMIT})',
                 fontsize=14, fontweight='bold')

    for idx, day in enumerate(days):
        d   = df[df['day']==day].copy()
        pos = simulate_mm(d, fv_fn, take_edge=2.0, pos_limit=POS_LIMIT)

        ax = axes[idx][0]
        ax.plot(d['timestamp'].values, pos.values, color='steelblue', lw=0.8)
        ax.axhline( POS_LIMIT, color='red', lw=1.5, ls='--', label='+limit')
        ax.axhline(-POS_LIMIT, color='red', lw=1.5, ls='--', label='-limit')
        ax.axhline(0,          color='black', lw=1)
        ax.set_title(f'Day {day}')
        ax.set_xlabel('Timestamp'); ax.set_ylabel('Position')
        ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

        limit_hits = ((pos>=POS_LIMIT) | (pos<=-POS_LIMIT)).sum()
        print(f'{prod} Day {day}: at-limit {limit_hits/len(pos)*100:.2f}% of ticks  '
              f'({limit_hits} ticks)')

    plt.tight_layout(); plt.show()
    print()

---
## Section 9 — Product Correlation

Checks whether EMERALDS and TOMATOES prices move together (and whether one leads the other).

- **r > 0.7** → products are correlated; fair value of one may inform the other.
- **Lag correlation**: if product A's return at t correlates with product B's return at t+k, A "leads" B by k ticks — a statistical arb opportunity.
- **r ≈ 0** → treat as independent products; no cross-product signal.

This also validates that our per-product P&L estimates are not double-counting correlated exposures.

In [ ]:
prod_names = list(products.keys())
if len(prod_names) < 2:
    print('Need ≥2 products for correlation analysis.')
else:
    series = {p: products[p].groupby('timestamp')['mid_price'].mean() for p in prod_names}
    aligned = pd.DataFrame(series).dropna()
    normed  = (aligned - aligned.mean()) / aligned.std()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Product Correlation', fontsize=14, fontweight='bold')

    for i, p in enumerate(aligned.columns):
        axes[0].plot(aligned.index, normed[p],
                     color=DAY_COLOURS[i%len(DAY_COLOURS)], lw=0.8, label=p, alpha=0.8)
    axes[0].set_title('Normalised Mid Prices')
    axes[0].set_xlabel('Timestamp'); axes[0].set_ylabel('Z-Score')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    p1, p2 = prod_names[0], prod_names[1]
    corr = aligned[p1].corr(aligned[p2])
    axes[1].scatter(aligned[p1], aligned[p2], s=3, alpha=0.3, color='steelblue')
    axes[1].set_title(f'Scatter {p1} vs {p2}  (r={corr:.4f})')
    axes[1].set_xlabel(p1); axes[1].set_ylabel(p2)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout(); plt.show()

    rets1 = aligned[p1].pct_change().dropna()
    rets2 = aligned[p2].pct_change().dropna()
    print(f'Contemporaneous r({p1},{p2}) = {corr:.4f}')
    print(f'Lag correlations ({p1} leads {p2}):')
    for lag in [0,1,2,3,5,10]:
        lc = rets1.corr(rets2) if lag==0 else rets1.iloc[:-lag].corr(rets2.iloc[lag:])
        print(f'  lag {lag:>2d}: {lc:.4f}')

    verdict = ('CORRELATED' if abs(corr)>0.7 else
               'WEAKLY CORRELATED' if abs(corr)>0.3 else 'INDEPENDENT')
    print(f'\n*** {verdict} (r={corr:.4f}) ***')

---
## Section 10 — Strategy Recommendation Summary

Consolidates all findings into a single actionable table.

**Column guide:**
- **Type** – STABLE (std/mean < 0.1%) / DRIFTING (0.1–1%) / VOLATILE (>1%)
- **Strategy** – which market-making or directional strategy to apply
- **Fair Value** – how to compute the reference price in the bot
- **Take Edge** – how many ticks from fair value before aggressively crossing the spread
- **Passive Edge** – how many ticks from fair value to post resting quotes
- **AC(lag=1)** – lag-1 return autocorrelation (negative = mean reverting)
- **Best Signal** – the signal with the highest |correlation| to next-tick direction
- **Mean Spread** – average bid-ask spread in ticks (= max round-trip revenue)

In [ ]:
rows = []
for prod, df in products.items():
    mp     = df['mid_price']
    std    = mp.std()
    spread = (df['ask_price_1'] - df['bid_price_1'])
    mean_spread = spread.mean()
    cv = std / mp.mean()

    if cv < 0.001:
        prod_type = 'STABLE'
        strategy  = 'Fixed FV market maker'
        fv_method = f'Constant @ {mp.mean():.0f}'
        take_edge = max(1, round(mean_spread*0.25))
        pass_edge = max(1, round(mean_spread*0.40))
    elif cv < 0.01:
        prod_type = 'DRIFTING'
        strategy  = 'Rolling FV market maker'
        fv_method = 'Rolling mean (20 ticks)'
        take_edge = max(1, round(mean_spread*0.30))
        pass_edge = max(1, round(mean_spread*0.45))
    else:
        prod_type = 'VOLATILE'
        strategy  = 'Mean-reversion / trend'
        fv_method = 'RM50 or external signal'
        take_edge = max(2, round(mean_spread*0.35))
        pass_edge = max(2, round(mean_spread*0.50))

    rets = mp.diff().dropna()
    ac1  = autocorr_at_lag(rets, 1)

    d2 = df.copy()
    d2['ret']         = d2['mid_price'].diff()
    d2['rm20']        = d2['mid_price'].rolling(20).mean()
    d2['z20']         = d2['mid_price'] - d2['rm20']
    d2['micro']       = compute_microprice(d2)
    d2['micro_delta'] = d2['micro'] - d2['mid_price']
    d2['next_ret']    = d2['ret'].shift(-1)
    sigs = [
        signal_stats(d2['ret'],         d2['next_ret'], 'ret1'),
        signal_stats(d2['z20'],         d2['next_ret'], 'z20'),
        signal_stats(d2['micro_delta'], d2['next_ret'], 'microprice_delta'),
    ]
    best = max(sigs, key=lambda s: abs(s['corr']) if not np.isnan(s['corr']) else 0)

    rows.append({
        'Product'      : prod,
        'Type'         : prod_type,
        'Strategy'     : strategy,
        'Fair Value'   : fv_method,
        'Take Edge'    : take_edge,
        'Passive Edge' : pass_edge,
        'AC(lag=1)'    : round(ac1, 4),
        'Best Signal'  : best['name'],
        'Signal Corr'  : round(best['corr'], 4),
        'Mean Spread'  : round(mean_spread, 4),
    })

rec = pd.DataFrame(rows).set_index('Product')
print('=== STRATEGY RECOMMENDATION SUMMARY ===\n')
print(rec.T.to_string())
print('\nKey:')
print('  Take Edge    – ticks from FV to aggressively cross the spread')
print('  Passive Edge – ticks from FV to post resting quotes')
print('  AC(lag=1)    – lag-1 autocorrelation (<0 = mean-reverting)')
print('  Best Signal  – highest |corr| signal; negative corr = use as contrarian')